# 03 — DINOv2 Training — DINOv2-ViT-B/14 (CRC Histology)

Fine-tuning **DINOv2-ViT-B/14** (SSL pretrained) on CRC histology tiles (9 classes).

- **Train/Val**: NCT-CRC-HE-100K (stratified split)
- **Test**: CRC-VAL-HE-7K (external test set)

## 0. Kaggle Setup

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical

In [ ]:
!pip install -q timm albumentations loguru
!pip install -q PyDrive2

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## 0b. Dataset (Kaggle input)

Provide the CRC histology datasets as Kaggle inputs and set the roots below:
- NCT-CRC-HE-100K for train/val
- CRC-VAL-HE-7K for final testing

In [ ]:
from __future__ import annotations

from pathlib import Path


def first_existing_dir(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    raise FileNotFoundError(
        "None of the candidate dataset directories exist:\n"
        + "\n".join([f"- {c}" for c in candidates])
    )


# ---- Update these if your Kaggle dataset names differ ----
# Common layouts:
# - Separate inputs: /kaggle/input/nct-crc-he-100k/* and /kaggle/input/crc-val-he-7k/*

trainval_candidates = [
    Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K")]
test_candidates = [
    Path("/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"),
]

TRAINVAL_ROOT = first_existing_dir(trainval_candidates)
TEST_ROOT = first_existing_dir(test_candidates)

print(f"Train/Val root: {TRAINVAL_ROOT}")
print(f"Test root     : {TEST_ROOT}")

## 1. Setup & Dependencies

In [ ]:
import os
import sys
import json
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = '/kaggle/working/xai-vit-medical'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import (
    CRCHistologyDataset,
    CRCSplits,
    DEFAULT_CRC_CLASSES,
    build_crc_transforms,
)
from src.utils.seed import set_seed

SEED = 42
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Dependencies loaded.')

## 2. Configuration

In [ ]:
# ---- Paths ----
SAVE_DIR = '/kaggle/working/xai-vit-medical/outputs/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Dataset ----
IMAGE_SIZE  = 224
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)
VAL_RATIO   = 0.25
NUM_WORKERS = 2

# ---- Phase 1 : Linear Probe (backbone gelé) ----
P1_EPOCHS = 5
P1_LR     = 1e-3
P1_WD     = 1e-4

# ---- Phase 2 : Fine-tuning (derniers blocs dégelés) ----
P2_EPOCHS     = 5
P2_LR         = 5e-5
P2_WD         = 0.05
P2_UNFREEZE_N = 4
WARMUP_EPOCHS = 2   # < P2_EPOCHS so cosine decay has time to run

# ---- Commun ----
BATCH_SIZE      = 32
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA     = 0.2

print('DINOv2 Configuration:')
for k, v in [
    ('Image size',       f'{IMAGE_SIZE}×{IMAGE_SIZE} (patch=14)'),
    ('Num classes',      NUM_CLASSES),
    ('Phase 1 epochs',   f'{P1_EPOCHS}  LR={P1_LR}  (linear probe)'),
    ('Phase 2 epochs',   f'{P2_EPOCHS}  LR={P2_LR}  (fine-tune {P2_UNFREEZE_N} blocs)'),
    ('Warmup epochs',    WARMUP_EPOCHS),
    ('Batch size',       BATCH_SIZE),
    ('Label smooth.',    LABEL_SMOOTHING),
    ('Mixup α',          MIXUP_ALPHA),
 ]:
    print(f'  {k:18s}: {v}')

## 3. Dataset & DataLoaders

In [ ]:
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=VAL_RATIO,
    random_state=SEED,
 )

train_dataset = CRCHistologyDataset(split='train', splits=crc_splits, image_size=IMAGE_SIZE)

train_dataset_clean = copy.copy(train_dataset)
train_dataset_clean.transform = build_crc_transforms(image_size=IMAGE_SIZE, split='val')

val_dataset  = CRCHistologyDataset(split='val',  splits=crc_splits, image_size=IMAGE_SIZE)
test_dataset = CRCHistologyDataset(split='test', splits=crc_splits, image_size=IMAGE_SIZE)

total = len(train_dataset)
class_counts = Counter(train_dataset.labels.tolist())
print(f'Train : {total} | Val : {len(val_dataset)} | Test : {len(test_dataset)}')
print('\nDistribution (train):')
for i, name in enumerate(CLASS_NAMES):
    cnt = class_counts.get(i, 0)
    bar = '█' * max(cnt // 500, 0)
    print(f'  {name:5s}: {cnt:>5d}  ({cnt/total*100:.1f}%)  {bar}')

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, drop_last=True,
 )
train_loader_clean = DataLoader(
    train_dataset_clean, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
 )
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
 )
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
 )
print(f'\nTrain batches       : {len(train_loader)}')
print(f'Train(clean) batches : {len(train_loader_clean)}')

## 4. Model — DINOv2Classifier

DINOv2 is loaded from `torch.hub` (`facebookresearch/dinov2`).
We add a linear classification head on top of the CLS token.

```
DINOv2-ViT-B/14
  backbone (12 transformer blocks, embed_dim=768)
    → CLS token  [B, 768]
  head: Dropout → Linear(768, 9)
```

In [ ]:
class DINOv2Classifier(nn.Module):
    """DINOv2-ViT-B/14 backbone + classification head.

    Chargé depuis torch.hub (facebookresearch/dinov2).
    Supporte 3 modes : frozen_linear_probe, partial_finetune, full_finetune.
    """

    EMBED_DIM = 768   # ViT-B embed dim

    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
        head_dropout: float = 0.1,
        hub_model: str = 'dinov2_vitb14',
    ) -> None:
        super().__init__()
        self.backbone = torch.hub.load(
            'facebookresearch/dinov2', hub_model, pretrained=True
        )
        self.head = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(self.EMBED_DIM, num_classes),
        )
        # Initialisation He pour la couche linéaire
        nn.init.trunc_normal_(self.head[-1].weight, std=0.02)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        cls_token = self.backbone(x)   # [B, 768] — token CLS
        return self.head(cls_token)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Retourne les features du token CLS (pour SAE, XAI)."""
        return self.backbone(x)

    def freeze_backbone(self) -> None:
        """Phase 1 : geler tout le backbone, seule la tête est entraînable."""
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n: int = P2_UNFREEZE_N) -> None:
        """Phase 2 : dégeler les n derniers blocs transformer + norm finale."""
        # Geler tout d'abord
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Dégeler les n derniers blocs
        total_blocks = len(self.backbone.blocks)
        for block in self.backbone.blocks[total_blocks - n:]:
            for p in block.parameters():
                p.requires_grad = True
        # Dégeler la norm finale
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        # La tête est toujours entraînable
        for p in self.head.parameters():
            p.requires_grad = True

    def unfreeze_all(self) -> None:
        """Dégeler tout le modèle (full fine-tuning)."""
        for p in self.parameters():
            p.requires_grad = True

    def count_trainable(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_total(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ---- Sanity check ----
print('Loading DINOv2-ViT-B/14 from torch.hub ...')
_m = DINOv2Classifier(num_classes=NUM_CLASSES)
print(f'  Total params     : {_m.count_total():,}')

_m.freeze_backbone()
print(f'  Trainable (frozen backbone) : {_m.count_trainable():,}')

_m.unfreeze_last_n_blocks(P2_UNFREEZE_N)
print(f'  Trainable (last {P2_UNFREEZE_N} blocks)  : {_m.count_trainable():,}')

_x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _m(_x).shape == (2, NUM_CLASSES), 'Shape mismatch'
del _m, _x
torch.cuda.empty_cache()
print('Architecture OK.')

## 5. Training Components

In [ ]:
def mixup_batch(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = MIXUP_ALPHA,
    num_classes: int = NUM_CLASSES,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if alpha <= 0:
        return images, F.one_hot(labels, num_classes).float(), labels
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    lbl = lam * F.one_hot(labels, num_classes).float() + (1 - lam) * F.one_hot(
        labels[idx], num_classes
    ).float()
    return mixed, lbl, labels


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    use_mixup: bool = True,
) -> tuple[float, float]:
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  Train', leave=False)

    for batch in pbar:
        images, labels = batch[0], batch[1]
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_mixup:
            images, mixed_labels, orig_labels = mixup_batch(images, labels)
            mixed_labels = mixed_labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            if use_mixup:
                loss = -(mixed_labels * F.log_softmax(outputs, dim=1)).sum(dim=1).mean()
            else:
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        ref = orig_labels if use_mixup else labels
        correct += preds.eq(ref).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    max_batches: int | None = None,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for batch_idx, batch in enumerate(
        tqdm(loader, desc='  Eval ', leave=False), start=1
    ):
        images, labels = batch[0], batch[1]
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if max_batches is not None and batch_idx >= max_batches:
            break

    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def make_adamw(model: nn.Module, lr: float, weight_decay: float) -> optim.AdamW:
    no_decay = {'bias', 'norm', 'cls_token', 'pos_embed', 'mask_token'}
    return optim.AdamW(
        [
            {'params': [p for n, p in model.named_parameters()
                        if not any(nd in n for nd in no_decay) and p.requires_grad],
             'weight_decay': weight_decay},
            {'params': [p for n, p in model.named_parameters()
                        if any(nd in n for nd in no_decay) and p.requires_grad],
             'weight_decay': 0.0},
        ],
        lr=lr,
    )


def make_scheduler(
    optimizer: optim.Optimizer, epochs: int, lr: float, warmup: int = WARMUP_EPOCHS,
) -> SequentialLR:
    w = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup)
    c = CosineAnnealingLR(optimizer, T_max=max(epochs - warmup, 1), eta_min=lr * 0.01)
    return SequentialLR(optimizer, [w, c], milestones=[warmup])


def run_training_phase(
    model: nn.Module,
    phase_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    lr: float,
    weight_decay: float,
    device: torch.device = DEVICE,
    use_mixup: bool = True,
    ckpt_path: str = '',
    train_clean_loader: DataLoader | None = None,
    train_clean_max_batches: int | None = 200,
) -> tuple[nn.Module, dict]:
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = make_adamw(model, lr, weight_decay)
    scheduler = make_scheduler(optimizer, epochs, lr)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    best_val_acc = 0.0
    history = {
        'train_loss': [], 'train_acc': [],
        'train_clean_loss': [], 'train_clean_acc': [],
        'val_loss': [], 'val_acc': [],
    }
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'\n{"="*68}')
    print(f'  {phase_name}  |  {epochs} epochs  |  lr={lr}  |  wd={weight_decay}')
    print(f'  Trainable params : {trainable:,}  |  mixup={use_mixup}')
    print(f'{"="*68}')

    for epoch in range(1, epochs + 1):
        t_obj_loss, t_obj_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, use_mixup
        )
        v_loss, v_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if train_clean_loader is not None:
            t_clean_loss, t_clean_acc, _, _ = evaluate(
                model, train_clean_loader, criterion, device,
                max_batches=train_clean_max_batches,
            )
        else:
            t_clean_loss, t_clean_acc = float('nan'), float('nan')

        history['train_loss'].append(t_obj_loss)
        history['train_acc'].append(t_obj_acc)
        history['train_clean_loss'].append(t_clean_loss)
        history['train_clean_acc'].append(t_clean_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        lr_now = optimizer.param_groups[0]['lr']
        tag = ''
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            if ckpt_path:
                torch.save({
                    'epoch': epoch,
                    'phase': phase_name,
                    'model_state_dict': model.state_dict(),
                    'val_loss': v_loss,
                    'val_acc': v_acc,
                    'history': history,
                    'class_names': CLASS_NAMES,
                }, ckpt_path)
            tag = ' ★'

        if train_clean_loader is not None:
            train_msg = (
                f'train_obj={t_obj_loss:.4f}/{t_obj_acc:.4f} | '
                f'train_clean={t_clean_loss:.4f}/{t_clean_acc:.4f}'
            )
        else:
            train_msg = f'train={t_obj_loss:.4f}/{t_obj_acc:.4f}'

        print(
            f'  Epoch {epoch:3d}/{epochs} | {train_msg} | '
            f'val={v_loss:.4f}/{v_acc:.4f} | lr={lr_now:.2e}{tag}'
        )

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'\n  Best val_acc={best_val_acc:.4f} restored from {ckpt_path}')

    return model, history


print('Training components defined.')

## 6. Phase 1 — Linear Probe (backbone gelé)

Seule la tête de classification est entraînée. 
Objectif : calibrer la tête sans toucher aux features SSL de DINOv2.

In [ ]:
print('=== PHASE 1 : Linear Probe ===')
print('Loading DINOv2-ViT-B/14 ...')

dinov2 = DINOv2Classifier(num_classes=NUM_CLASSES, head_dropout=0.1).to(DEVICE)
dinov2.freeze_backbone()

print(f'Backbone gelé. Trainable : {dinov2.count_trainable():,} / {dinov2.count_total():,}')

dinov2, p1_history = run_training_phase(
    model=dinov2,
    phase_name='Phase 1 — Linear Probe',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=P1_EPOCHS,
    lr=P1_LR,
    weight_decay=P1_WD,
    device=DEVICE,
    use_mixup=False,
    ckpt_path=os.path.join(SAVE_DIR, 'dinov2_phase1_best.pth'),
    train_clean_loader=train_loader_clean,
    train_clean_max_batches=200,
)

print('\nPhase 1 terminée.')

## 7. Phase 2 — Fine-tuning (derniers blocs dégelés)

On dégèle les `P2_UNFREEZE_N` derniers blocs transformer + norm finale.
LR très faible (5e-5) pour préserver les features SSL apprises.

In [ ]:
print(f'=== PHASE 2 : Fine-tuning (derniers {P2_UNFREEZE_N} blocs) ===')

dinov2.unfreeze_last_n_blocks(P2_UNFREEZE_N)
print(f'Blocs dégelés. Trainable : {dinov2.count_trainable():,} / {dinov2.count_total():,}')

dinov2, p2_history = run_training_phase(
    model=dinov2,
    phase_name=f'Phase 2 — Fine-tune last {P2_UNFREEZE_N} blocks',
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=P2_EPOCHS,
    lr=P2_LR,
    weight_decay=P2_WD,
    device=DEVICE,
    use_mixup=True,
    ckpt_path=os.path.join(SAVE_DIR, 'dinov2_best.pth'),
    train_clean_loader=train_loader_clean,
    train_clean_max_batches=200,
)

print('\nPhase 2 terminée.')

In [ ]:
# Upload checkpoint final vers Google Drive
for fname in ['dinov2_phase1_best.pth', 'dinov2_best.pth']:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        f = drive.CreateFile({'title': fname})
        f.SetContentFile(fpath)
        f.Upload()
        print(f'Uploaded: {fname}  (id={f["id"]})')

## 8. Evaluation & Results

In [ ]:
def plot_history_two_phases(h1: dict, h2: dict, model_name: str) -> None:
    n1 = len(h1['val_loss'])
    n2 = len(h2['val_loss'])
    ep1 = list(range(1, n1 + 1))
    ep2 = list(range(n1 + 1, n1 + n2 + 1))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(ep1, h1['train_clean_loss'], 'b-',  label='Train(clean) P1')
    ax1.plot(ep1, h1['val_loss'],         'r-',  label='Val P1')
    ax1.plot(ep2, h2['train_clean_loss'], 'b--', label='Train(clean) P2')
    ax1.plot(ep2, h2['val_loss'],         'r--', label='Val P2')
    ax1.axvline(n1, color='gray', linestyle=':', label='Phase 1→2')
    ax1.set(xlabel='Epoch', ylabel='Loss', title=f'{model_name} — Loss')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    ax2.plot(ep1, h1['train_clean_acc'], 'b-',  label='Train(clean) P1')
    ax2.plot(ep1, h1['val_acc'],         'r-',  label='Val P1')
    ax2.plot(ep2, h2['train_clean_acc'], 'b--', label='Train(clean) P2')
    ax2.plot(ep2, h2['val_acc'],         'r--', label='Val P2')
    ax2.axvline(n1, color='gray', linestyle=':', label='Phase 1→2')
    ax2.set(xlabel='Epoch', ylabel='Accuracy', title=f'{model_name} — Accuracy')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(SAVE_DIR, f'{model_name}_curves.png')
    plt.savefig(out, dpi=150)
    plt.show()

    gap = float(np.nanmax(h2['train_clean_acc']) - np.nanmax(h2['val_acc']))
    status = 'overfitting detecté' if gap > 0.10 else 'gap acceptable'
    print(
        f'  Phase 2 — best_train_clean={np.nanmax(h2["train_clean_acc"]):.4f}  '
        f'best_val={np.nanmax(h2["val_acc"]):.4f}  gap={gap:.4f} ({status})'
    )


def evaluate_full(
    model: nn.Module,
    loader: DataLoader,
    model_name: str,
    split: str = 'val',
    device: torch.device = DEVICE,
) -> dict:
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    loss, acc, preds, labels = evaluate(model, loader, criterion, device)
    print(f'\n{model_name} — {split.upper()}')
    print(f'  Loss     : {loss:.4f}')
    print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set(xlabel='Predicted', ylabel='True',
           title=f'{model_name} — Confusion Matrix ({split})')
    plt.tight_layout()
    out = os.path.join(SAVE_DIR, f'{model_name}_cm_{split}.png')
    plt.savefig(out, dpi=150)
    plt.show()
    return {'loss': loss, 'acc': acc, 'preds': preds, 'labels': labels}


# ---- Courbes des 2 phases ----
plot_history_two_phases(p1_history, p2_history, 'DINOv2-ViT-B_14')

In [ ]:
# ---- Évaluation finale (meilleur checkpoint Phase 2) ----
ckpt = torch.load(os.path.join(SAVE_DIR, 'dinov2_best.pth'), map_location=DEVICE)
dinov2.load_state_dict(ckpt['model_state_dict'])
print(f"Best checkpoint: phase={ckpt['phase']}  epoch={ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.4f}  val_acc={ckpt['val_acc']:.4f}")

dino_val_results  = evaluate_full(dinov2, val_loader,  'DINOv2-ViT-B_14', 'val')
dino_test_results = evaluate_full(dinov2, test_loader, 'DINOv2-ViT-B_14', 'test')

## 9. Save Summary & Upload

In [ ]:
summary = {
    'dinov2_vitb14': {
        'val_loss':       float(dino_val_results['loss']),
        'val_acc':        float(dino_val_results['acc']),
        'test_loss':      float(dino_test_results['loss']),
        'test_acc':       float(dino_test_results['acc']),
        'best_epoch':     int(ckpt['epoch']),
        'best_phase':     ckpt['phase'],
        'p1_epochs':      len(p1_history['train_loss']),
        'p2_epochs':      len(p2_history['train_loss']),
        'unfreeze_n_blocks': P2_UNFREEZE_N,
    }
}

summary_path = os.path.join(SAVE_DIR, 'dino_training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

m = summary['dinov2_vitb14']
print('DINOv2 Training Summary:')
print(f'  val_acc   = {m["val_acc"]:.4f}')
print(f'  test_acc  = {m["test_acc"]:.4f}')
print(f'  Phase 1   : {m["p1_epochs"]} epochs (linear probe)')
print(f'  Phase 2   : {m["p2_epochs"]} epochs (fine-tune {m["unfreeze_n_blocks"]} blocs)')
print(f'  Best epoch: {m["best_epoch"]} ({m["best_phase"]})')

In [ ]:
files_to_upload = [
    'dino_training_summary.json',
    'DINOv2-ViT-B_14_curves.png',
    'DINOv2-ViT-B_14_cm_val.png',
    'DINOv2-ViT-B_14_cm_test.png',
]

for fname in files_to_upload:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        f = drive.CreateFile({'title': fname})
        f.SetContentFile(fpath)
        f.Upload()
        print(f'Uploaded: {fname}  (id={f["id"]})')
    else:
        print(f'Skipped: {fname}')

print('\nDone. Tous les checkpoints prêts pour 04_xai_classical.ipynb')